Dependencies

In [ ]:
%%bash
pip install -q "qdrant-client[fastembed]>=1.14.2" sentence-transformers transformers accelerate pydantic gradio pandas numpy tqdm

In [ ]:
import os, glob, shutil, subprocess, time, urllib.request, re
import pandas as pd
from qdrant_client import QdrantClient, models

WORK_STORAGE = "/kaggle/working/qdrant_storage"
WORK_BIN = "/kaggle/working/qdrant_bin/qdrant"
WORK_ARTIFACTS = "/kaggle/working/artifacts"

# If running in a fresh Notebook 2 with Notebook 1 attached in /kaggle/input/, copy writable storage to /kaggle/working
if not os.path.exists(WORK_STORAGE):
    in_storage = glob.glob("/kaggle/input/**/qdrant_storage", recursive=True)
    in_bin = glob.glob("/kaggle/input/**/qdrant_bin/qdrant", recursive=True)
    in_art = glob.glob("/kaggle/input/**/artifacts", recursive=True)
    assert in_storage and in_art, "Cannot find qdrant_storage/artifacts in /kaggle/working or /kaggle/input!"
    print("Copying Qdrant storage from input to writable /kaggle/working...")
    shutil.copytree(in_storage[0], WORK_STORAGE)
    shutil.copytree(in_art[0], WORK_ARTIFACTS, dirs_exist_ok=True)
    os.makedirs("/kaggle/working/qdrant_bin", exist_ok=True)
    if in_bin:
        shutil.copy2(in_bin[0], WORK_BIN)
    else:
        subprocess.run(
            "wget -q https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz -O /tmp/q.tar.gz && tar -xzf /tmp/q.tar.gz -C /kaggle/working/qdrant_bin",
            shell=True, check=True,
        )
    os.chmod(WORK_BIN, 0o755)

with open("/kaggle/working/qdrant_config.yaml", "w") as f:
    f.write("storage:\n  storage_path: /kaggle/working/qdrant_storage\nservice:\n  http_port: 6333\n  grpc_port: 6334\ntelemetry_disabled: true\n")

In [ ]:
# Start Qdrant if not already running
def qdrant_alive():
    try:
        with urllib.request.urlopen("http://localhost:6333/healthz", timeout=1) as r:
            return r.status == 200
    except Exception:
        return False

if not qdrant_alive():
    qdrant_proc = subprocess.Popen(
        [WORK_BIN, "--config-path", "/kaggle/working/qdrant_config.yaml"],
        stdout=open("/tmp/qdrant2.log", "w"),
        stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        if qdrant_alive():
            break
        time.sleep(1)
    assert qdrant_alive(), "Qdrant failed to start. Check /tmp/qdrant2.log"

client = QdrantClient(url="http://localhost:6333", prefer_grpc=True, timeout=60)
COLLECTION_NAME = "recipes"
print("Qdrant UP! Points:", client.get_collection(COLLECTION_NAME).points_count)

# --- Title-Consistency Safety Patch (catches omitted ingredients like 'on Toast') ---
df_clean = pd.read_parquet(f"{WORK_ARTIFACTS}/recipes_clean.parquet")

TITLE_GLUTEN_RE = re.compile(r"\b(toast|bread|sandwich|bagel|pasta|spaghetti|macaroni|pizza|noodle|noodles|ramen|udon|tortilla|pita|biscuit|waffle|pancake|muffin|cake|brownie|cookie|pie|beer|ale|croissant|brioche|couscous|bulgur|seitan|soy sauce)\b", re.I)
TITLE_GF_EXEMPT_RE = re.compile(r"\b(gluten[- ]free|flourless|keto|cauliflower|zucchini|rice noodle|rice paper|corn tortilla)\b", re.I)
TITLE_HALAL_RE = re.compile(r"\b(pork|bacon|ham|prosciutto|pancetta|pepperoni|wine|beer|bourbon|whiskey|rum|vodka|brandy|champagne|sake|mirin)\b", re.I)
TITLE_HALAL_EXEMPT_RE = re.compile(r"\b(turkey bacon|beef bacon|non[- ]alcoholic|ginger ale|root beer|vinegar)\b", re.I)
TITLE_VEGAN_RE = re.compile(r"\b(chicken|beef|pork|turkey|lamb|salmon|tuna|shrimp|bacon|ham|egg|eggs|cheese|feta|cheddar|parmesan|mozzarella|butter|cream|milk|yogurt|honey|mayo)\b", re.I)
TITLE_VEGAN_EXEMPT_RE = re.compile(r"\b(vegan|plant[- ]based|dairy[- ]free|coconut milk|almond milk|soy milk|oat milk|peanut butter|butternut)\b", re.I)
TITLE_NUT_RE = re.compile(r"\b(peanut|almond|walnut|pecan|cashew|pistachio|hazelnut|macadamia|pesto|nutella|praline)\b", re.I)

patched = 0
for row in df_clean.itertuples(index=False):
    title = str(row.Name)
    tags = list(row.diet_tags)
    conf = dict(row.diet_confidence)
    changed = False

    if "gluten_free" in tags and TITLE_GLUTEN_RE.search(title) and not TITLE_GF_EXEMPT_RE.search(title):
        tags.remove("gluten_free"); conf["gluten_free"] = "unknown"; changed = True
    if "halal_friendly" in tags and TITLE_HALAL_RE.search(title) and not TITLE_HALAL_EXEMPT_RE.search(title):
        tags.remove("halal_friendly"); conf["halal_friendly"] = "unknown"; changed = True
    if "vegan" in tags and TITLE_VEGAN_RE.search(title) and not TITLE_VEGAN_EXEMPT_RE.search(title):
        tags.remove("vegan"); conf["vegan"] = "unknown"; changed = True
    if "nut_free" in tags and TITLE_NUT_RE.search(title):
        tags.remove("nut_free"); conf["nut_free"] = "unknown"; changed = True

    if changed:
        client.set_payload(
            collection_name=COLLECTION_NAME,
            payload={"diet_tags": tags, "diet_confidence": conf},
            points=[int(row.RecipeId)],
            wait=False,
        )
        patched += 1

print(f"Title-consistency safety patch downgraded {patched:,} ambiguous/incomplete recipes.")

Loading encoders and LLM

In [ ]:
import json, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from fastembed import SparseTextEmbedding
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running inference on:", device)

with open(f"{WORK_ARTIFACTS}/taxonomy.json") as f:
    TAXONOMY = json.load(f)
SYNONYMS = TAXONOMY["synonyms"]
KNOWN_VOCAB = set(TAXONOMY["known_vocab"])

dense_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
reranker_model = CrossEncoder("BAAI/bge-reranker-base", device=device, max_length=256)

# Small open-weights GPU LLM for complex query fallback (~6 GB VRAM fp16)
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
llm_model.eval()
print("All models loaded successfully!")